# Week 9 Activity: APIs and Web Scraping 

# Setup

In [3]:
## imports
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup  
import yaml
import os

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## census wrapper and helper package with state FIPS codes
## if not installed: python -m pip install census us
from census import Census
from us import states

## print all of text
pd.options.display.max_colwidth = 1000

## Activity 1: NAEP API (no credentials needed)

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

A. Using their example query of 2011 writing scores separated by gender
based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string

Use the ( ) syntax to formulate a long string stored as `example_naep_query` without linebreaks added

In [ ]:
## your code here


B. Use requests to call the API and print the response

In [ ]:
## your code here


C. Get the json contents of the response

In [ ]:
## your code here


D. Extract the result from that json and turn it into a dataframe

In [ ]:
## your code here

E. Run code to see what happens with an error in the query

In [1]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [4]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
## uncomment to see error
# naep_wrong_resp.json()

<Response [400]>

In [5]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except:
    pass

In [7]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along thos elines
naep_wrong_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

'{"statusCode":400,"result": "System.Exception: The query \'SELECT DISTINCT Framework FROM Cycles WHERE Subject=\'WRI\' AND Cohort=2 AND CONVERT(VARCHAR(10),Year)+Sample IN (\'2025R3\')\' did not return exactly 1 framework. Make sure you can trend the years defined for the given subject and cohort.\r\n   at NRCDataService3.GetAdhocData.GetFramework(NDEContext& ndeContext, String subjectCode, List`1 yearSamples, String cohort) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2733\r\n   at NRCDataService3.GetAdhocData.PopulateBaseOrchestratorRequest() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 2348\r\n   at NRCDataService3.GetAdhocData.ConstructRequest_Datapoint() in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 947\r\n   at NRCDataService3.GetAdhocData.Page_Load(Object sender, EventArgs e) in C:\\projects\\ndecore2025\\NRCDataService2\\GetAdhocData.aspx.cs:line 354"}'

NAEP results not found


F. Write a function to make repeated calls to the API

- Say we want to pull the data for grades 4, 8, and 12

Task: write a function called `pull_one_grade` that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [ ]:
## your code here 


## Activity 2: API for Yelp

In [ ]:
## load creds- STUDENTS: REPLACE WITH PATHNAME TO YOURS 
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

## get the key
API_KEY = creds['yelp_api']['api_key']

In [ ]:
## function you can use to clean the results
def clean_yelp_json(one_biz):
    """
    Function that cleans the json file with a successful call to the Yelp API
    Inputs:
    one_biz: json entry for one business
    Returns:
    df_str: a dataframe with the values the API returns that are strings
    """
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)


A. Try running a business search query for another place in the United States by constructing a query similar to `yelp_genquery` but changing the location parameter


In [ ]:
## your code here



B. Use the documentation for the business details endpoint to pull details about a specific business

Documentation: https://docs.developer.yelp.com/reference/v3_business_info

In [ ]:
## your code here

## Activity 3: Census API (credentials and a wrapper)

API documentation: https://www.census.gov/data/developers.html

Wrapper documentation: https://pypi.org/project/census/

Variable list (ACS 5-year, 2024): https://api.census.gov/data/2024/acs/acs5/variables.html

A. Load the yaml file with your API key for the Census and use the `Census` command to establish a connection

In [ ]:
## load creds- STUDENTS: REPLACE WITH PATHNAME TO YOURS
with open("../creds.yaml", 'r') as stream:
    try:
        creds = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

## get the key
CENSUS_KEY = creds['census_api']['api_key']

## use the key to create a connection to the API
## your code here

B. Pick a state and use the `census` wrapper to pull total population, median household income, and the two poverty variables for all counties in that state

| Variable | Description |
|:---|:---|
| `NAME` | Name of the geography |
| `B01003_001E` | Total population |
| `B19013_001E` | Median household income |
| `B17001_001E` | Population for whom poverty status is determined |
| `B17001_002E` | Population with income below the poverty level |

In [ ]:
## your code here

C. Construct the poverty rate and find the three counties in the state with the highest and lowest rates

In [ ]:
## your code here

D. Repeat the pull for 2019 using the `year` argument, merge the two years on the county FIPS code, and calculate the change in median household income

**Note**: for simplicity, we're just calculating the change in nominal income since each year's median income estimates are in that year's dollars; if comparing things like purchasing power, we'd want to adjust for inflation using an index like the CPI

In [ ]:
## your code here

## Activity 4: Scraping a static website with `requests` and `BeautifulSoup`

In this activity, we'll scrape news stories from Georgetown's news site: https://www.georgetown.edu/news/all-news/

We'll follow the scraping routine from lecture:

- **Step 1:** Find a website with information you want to collect
- **Step 2:** Understand the website (use Inspect or SelectorGadget)
- **Step 3:** Write code to collect one realization of the data (**Part A**)
- **Step 4:** Build a scraper: generalize your code into a function (**Parts B-D**)

Before starting, check the site's `robots.txt` (https://www.georgetown.edu/robots.txt) --- it only disallows WordPress admin pages, so the news pages are fair game. We'll still be kind to the server by pausing between requests.

A. Scrape one article

Use this article: https://www.georgetown.edu/news/chinese-language-joy-of-learning/

1. Use `requests` to get the page and check the status code
2. Create a `BeautifulSoup` object
3. Use Inspect or SelectorGadget to find CSS selectors for, and then extract:
    - the title of the article
    - the date it was published
    - the author
    - the text of the article (all paragraphs joined into one string)
4. Store the url and the four pieces of information in a one-row pandas DataFrame

**Hints:**

- If you `.select("p")`, you'll also grab paragraphs that aren't part of the story (e.g., the address in the page footer). Find a more specific selector for the story's body text
- When you extract the title, date, and author, you'll see some extra text like `"Title: "` that isn't visible on the page. Look at the html to see where it comes from, then remove it

In [ ]:
## your code here

B. Write a function to scrape one article

Write a function called `scrape_gu_article` that:

- Takes one input: the url of a Georgetown news article
- Returns a dictionary with the url, title, date, author, and text of the article

Then test your function on these two articles:

- https://www.georgetown.edu/news/chinese-language-joy-of-learning/
- https://www.georgetown.edu/news/board-directors-fall-2026/

**Hint:** not every article lists an author. What happens to your code from Part A when there's no `.author` element on the page? Write your function so it returns `None` for the author in that case rather than throwing an error.

In [ ]:
## your code here